# Duplicate Sensitivity Analysis

This experiment investigates how retaining or removing exact duplicate booking records affects Random Forest performance.

Both approaches are evaluated using the same group-aware cross-validation folds. Duplicate removal is applied only within each training fold to maintain a consistent validation dataset.

The held-out test set is not used.

## Cell 1 — Load Training Data

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the validated dataset and existing split
data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")

df = pd.read_csv(data_path)
splits = np.load(split_path)

# Use training records only
train_df = df.iloc[splits["train_idx"]].copy()

# Use the same 21 approved model features
excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]

X_train = train_df.drop(columns=excluded)
y_train = train_df["is_canceled"]

groups_train = pd.util.hash_pandas_object(X_train, index=False)

print("Training records:", len(train_df))
print("Model features:", X_train.shape[1])
print("Exact duplicate rows:", train_df.duplicated().sum())
print("Unique feature groups:", groups_train.nunique())


Training records: 95334
Model features: 21
Exact duplicate rows: 25616
Unique feature groups: 65232


## Configure Random Forest and Cross-Validation

In [2]:

import sys
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold

# Import shared preprocessing
repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))

from src.preprocessing.pipeline import build_preprocessor

# Use the same selected Random Forest configuration
rf_model = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        class_weight=None,
        random_state=42,
        n_jobs=2
    ))
])

# Use the same group-aware validation strategy
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Random Forest configured successfully.")
print("Cross-validation folds:", cv.n_splits)
print("Classification threshold: 0.40")


Random Forest configured successfully.
Cross-validation folds: 5
Classification threshold: 0.40


## Check Duplicates Within Each Training Fold

In [3]:

# Prepare the same validation folds for both approaches
cv_splits = list(cv.split(X_train, y_train, groups_train))

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    fold_train = train_df.iloc[train_idx]

    duplicates = fold_train.duplicated().sum()

    print(f"Fold {fold}")
    print(f"  Training records: {len(fold_train)}")
    print(f"  Exact duplicates: {duplicates}")
    print(f"  After deduplication: {len(fold_train) - duplicates}")
    print(f"  Validation records: {len(val_idx)}\n")


Fold 1
  Training records: 76445
  Exact duplicates: 20631
  After deduplication: 55814
  Validation records: 18889

Fold 2
  Training records: 76040
  Exact duplicates: 20317
  After deduplication: 55723
  Validation records: 19294

Fold 3
  Training records: 76981
  Exact duplicates: 21167
  After deduplication: 55814
  Validation records: 18353

Fold 4
  Training records: 76135
  Exact duplicates: 20384
  After deduplication: 55751
  Validation records: 19199

Fold 5
  Training records: 75735
  Exact duplicates: 19965
  After deduplication: 55770
  Validation records: 19599



## Evaluate Random Forest With Duplicates

In [4]:

from sklearn.base import clone
from sklearn.metrics import (
    precision_score, recall_score,
    f1_score, average_precision_score
)

# Approach A: Keep all duplicate records
kept_results = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    model = clone(rf_model)

    model.fit(
        X_train.iloc[train_idx],
        y_train.iloc[train_idx]
    )

    probabilities = model.predict_proba(X_train.iloc[val_idx])[:, 1]
    predictions = (probabilities >= 0.40).astype(int)
    actual = y_train.iloc[val_idx]

    kept_results.append({
        "Fold": fold,
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1": f1_score(actual, predictions),
        "PR-AUC": average_precision_score(actual, probabilities)
    })

    print(f"Fold {fold} completed.")

kept_results = pd.DataFrame(kept_results)
display(kept_results.round(4))


Fold 1 completed.
Fold 2 completed.
Fold 3 completed.
Fold 4 completed.
Fold 5 completed.


,Fold,Precision,Recall,F1,PR-AUC
0,1,0.7227,0.7549,0.7384,0.8463
1,2,0.7328,0.7423,0.7375,0.8494
2,3,0.7091,0.7427,0.7255,0.8263
3,4,0.6975,0.7465,0.7212,0.8324
4,5,0.7479,0.7400,0.7440,0.8558


## Evaluate Random Forest Without Duplicates

In [5]:

# Approach B: Remove exact duplicates from each training fold
dedup_results = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    model = clone(rf_model)

    # Identify duplicates using all original dataset columns
    keep_mask = ~train_df.iloc[train_idx].duplicated().to_numpy()

    X_fit = X_train.iloc[train_idx].iloc[keep_mask]
    y_fit = y_train.iloc[train_idx].iloc[keep_mask]

    model.fit(X_fit, y_fit)

    # Evaluate on the same validation records as Approach A
    probabilities = model.predict_proba(X_train.iloc[val_idx])[:, 1]
    predictions = (probabilities >= 0.40).astype(int)
    actual = y_train.iloc[val_idx]

    dedup_results.append({
        "Fold": fold,
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1": f1_score(actual, predictions),
        "PR-AUC": average_precision_score(actual, probabilities)
    })

    print(f"Fold {fold} completed.")

dedup_results = pd.DataFrame(dedup_results)
display(dedup_results.round(4))


Fold 1 completed.
Fold 2 completed.
Fold 3 completed.
Fold 4 completed.
Fold 5 completed.


,Fold,Precision,Recall,F1,PR-AUC
0,1,0.7770,0.6553,0.7110,0.8368
1,2,0.7717,0.6162,0.6852,0.8278
2,3,0.7523,0.5918,0.6625,0.7987
3,4,0.7694,0.6054,0.6776,0.8224
4,5,0.8021,0.5693,0.6660,0.8260


## Create the Final Comparison Table

In [6]:

# Compare the average performance of both approaches
metrics = ["Precision", "Recall", "F1", "PR-AUC"]

comparison = pd.DataFrame({
    "Keep Duplicates": kept_results[metrics].mean(),
    "Remove Duplicates": dedup_results[metrics].mean()
}).T

comparison.loc["Difference (Remove - Keep)"] = (
    comparison.loc["Remove Duplicates"]
    - comparison.loc["Keep Duplicates"]
)

display(comparison.round(4))


,Precision,Recall,F1,PR-AUC
Keep Duplicates,0.7220,0.7453,0.7333,0.8420
Remove Duplicates,0.7745,0.6076,0.6805,0.8223
Difference (Remove - Keep),0.0525,-0.1377,-0.0529,-0.0197


## Final Duplicate Sensitivity Conclusion


## Duplicate Sensitivity Analysis — Conclusion

The original hotel booking dataset contained 31,994 exact duplicate rows. Since the published dataset does not include unique reservation identifiers, it is not possible to determine whether these records represent separate bookings with identical attributes or accidental duplication.

To investigate their influence, a sensitivity experiment compared two approaches using the same five-fold group-aware validation splits and tuned Random Forest configuration:

1. Retaining all training records.
2. Removing exact duplicates from each training fold.

Both approaches were evaluated on identical validation records using a fixed classification threshold of 0.40.

| Metric | Keep Duplicates | Remove Duplicates |
|---|---:|---:|
| Precision | 0.7220 | 0.7745 |
| Recall | 0.7453 | 0.6076 |
| F1-score | 0.7333 | 0.6805 |
| PR-AUC | 0.8420 | 0.8223 |

Removing duplicates increased precision but reduced recall, F1-score, and PR-AUC on the original validation distribution.

These findings show that duplicate handling has a substantial influence on model performance. However, the experiment cannot establish whether identical records are genuine separate bookings or data errors. Furthermore, the fixed threshold was originally selected for the duplicate-retaining model.

**Decision:** Retain duplicate records in the primary modelling workflow to preserve the published booking-record distribution. Document the uncertainty surrounding duplicate identity and acknowledge that model performance is sensitive to duplicate handling.

